# Titanic Survival Prediction Model

## Kaggle Competition: Predict Titanic Survival

**Objective**: Build a machine learning model to predict which passengers survived the Titanic disaster.

**Current Score**: 0.80143 (Ranked 812)

---

## Section 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")

## Section 2: Load and Explore Data

In [ ]:
# Load datasets
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print("Training Data Shape:", train_df.shape)
print("Test Data Shape:", test_df.shape)
print("\n" + "="*50)
print("TRAINING DATA - First 5 rows:")
print(train_df.head())
print("\n" + "="*50)
print("DATASET INFO:")
print(train_df.info())

## Section 3: Exploratory Data Analysis (EDA)

In [ ]:
# 3.1 Missing Values Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Missing values in training data
missing_train = train_df.isnull().sum()
missing_train = missing_train[missing_train > 0].sort_values(ascending=False)

missing_train.plot(kind='barh', ax=axes[0], color='coral')
axes[0].set_title('Missing Values in Training Data', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Count')

# Missing values percentage
missing_pct = (train_df.isnull().sum() / len(train_df) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

missing_pct.plot(kind='barh', ax=axes[1], color='skyblue')
axes[1].set_title('Missing Values Percentage', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Percentage (%)')

plt.tight_layout()
plt.show()

print("Missing Values:")
print(missing_train)
print("\nPercentage:")
print(missing_pct)

In [ ]:
# 3.2 Survival Distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Survival count
survival_counts = train_df['Survived'].value_counts()
axes[0].bar(['Did Not Survive', 'Survived'], survival_counts.values, color=['#FF6B6B', '#4ECDC4'])
axes[0].set_title('Survival Distribution', fontweight='bold')
axes[0].set_ylabel('Count')

# Survival by Gender
sns.barplot(x='Sex', y='Survived', data=train_df, ax=axes[1], palette='Set2')
axes[1].set_title('Survival by Gender', fontweight='bold')
axes[1].set_ylabel('Survival Rate')

# Survival by Passenger Class
sns.barplot(x='Pclass', y='Survived', data=train_df, ax=axes[2], palette='Set1')
axes[2].set_title('Survival by Passenger Class', fontweight='bold')
axes[2].set_xlabel('Passenger Class')
axes[2].set_ylabel('Survival Rate')

plt.tight_layout()
plt.show()

print("\nSurvival Statistics:")
print(f"Overall Survival Rate: {train_df['Survived'].mean():.1%}")
print(f"\nSurvival by Gender:\n{train_df.groupby('Sex')['Survived'].mean()}")
print(f"\nSurvival by Class:\n{train_df.groupby('Pclass')['Survived'].mean()}")

In [ ]:
# 3.3 Age and Fare Analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Age distribution by survival
axes[0, 0].hist(train_df[train_df['Survived']==1]['Age'].dropna(), bins=20, alpha=0.6, label='Survived', color='green')
axes[0, 0].hist(train_df[train_df['Survived']==0]['Age'].dropna(), bins=20, alpha=0.6, label='Did Not Survive', color='red')
axes[0, 0].set_title('Age Distribution by Survival', fontweight='bold')
axes[0, 0].set_xlabel('Age')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].legend()

# Fare distribution by survival
axes[0, 1].hist(train_df[train_df['Survived']==1]['Fare'].dropna(), bins=20, alpha=0.6, label='Survived', color='green')
axes[0, 1].hist(train_df[train_df['Survived']==0]['Fare'].dropna(), bins=20, alpha=0.6, label='Did Not Survive', color='red')
axes[0, 1].set_title('Fare Distribution by Survival', fontweight='bold')
axes[0, 1].set_xlabel('Fare')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].legend()

# Age by Passenger Class
train_df.boxplot(column='Age', by='Pclass', ax=axes[1, 0])
axes[1, 0].set_title('Age by Passenger Class', fontweight='bold')
axes[1, 0].set_xlabel('Passenger Class')
axes[1, 0].set_ylabel('Age')

# Family size analysis
train_df['FamilySize'] = train_df['SibSp'] + train_df['Parch'] + 1
sns.barplot(x='FamilySize', y='Survived', data=train_df, ax=axes[1, 1])
axes[1, 1].set_title('Survival by Family Size', fontweight='bold')
axes[1, 1].set_ylabel('Survival Rate')

plt.tight_layout()
plt.show()

print("\nAge Statistics:")
print(train_df['Age'].describe())
print(f"\nMedian Age: {train_df['Age'].median()}")

## Section 4: Data Preprocessing

In [ ]:
# Combine train and test for preprocessing
df = pd.concat([train_df, test_df], ignore_index=True)

# 4.1 Handle Missing Values
# Age: Fill with median
df['Age'].fillna(df['Age'].median(), inplace=True)

# Embarked: Fill with mode
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# Fare: Fill with median
df['Fare'].fillna(df['Fare'].median(), inplace=True)

# 4.2 Feature Engineering
# Extract Title from Name
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Standardize titles
title_mapping = {
    'Mr': 'Mr', 'Mrs': 'Mrs', 'Miss': 'Miss', 'Master': 'Master',
    'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
    'Mlle': 'Miss', 'Countess': 'Rare', 'Ms': 'Miss', 'Lady': 'Rare',
    'Jonkheer': 'Rare', 'Sir': 'Rare', 'Dona': 'Rare', 'Mme': 'Mrs'
}
df['Title'] = df['Title'].map(title_mapping).fillna('Rare')

# Family Size
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Is Alone
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# 4.3 Drop unnecessary columns
df = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

print("✅ Data Preprocessing Complete!")
print(f"\nDataset Shape: {df.shape}")
print(f"\nMissing Values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"\nFirst few rows:")
print(df.head())

## Section 5: Feature Encoding and Scaling

In [ ]:
# 5.1 Encode Categorical Variables
# Sex: Male=0, Female=1
df['Sex'] = (df['Sex'] == 'female').astype(int)

# Title: Encode numerically
title_encoder = LabelEncoder()
df['Title'] = title_encoder.fit_transform(df['Title'])

# Embarked: Encode numerically
embarked_encoder = LabelEncoder()
df['Embarked'] = embarked_encoder.fit_transform(df['Embarked'])

print("✅ Categorical Variables Encoded!")
print(f"\nDataframe Info:")
print(df.dtypes)
print(f"\nSample Data:")
print(df.head())

In [ ]:
# 5.2 Prepare Data for Modeling
# Separate train and test
train_processed = df[:len(train_df)].copy()
test_processed = df[len(train_df):].copy()

# Separate features and target
X_train_full = train_processed.drop('Survived', axis=1)
y_train_full = train_processed['Survived']
X_test = test_processed.drop('Survived', axis=1)

# Train-validation split
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

# 5.3 Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Data Prepared and Scaled!")
print(f"\nTraining Set: {X_train_scaled.shape}")
print(f"Validation Set: {X_val_scaled.shape}")
print(f"Test Set: {X_test_scaled.shape}")
print(f"\nFeature Names: {list(X_train_full.columns)}")

## Section 6: Model Training and Evaluation

In [ ]:
# 6.1 Train Random Forest Classifier
print("Training Random Forest Classifier...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

# Predictions
rf_pred_train = rf_model.predict(X_train_scaled)
rf_pred_val = rf_model.predict(X_val_scaled)

# Evaluation
rf_train_acc = accuracy_score(y_train, rf_pred_train)
rf_val_acc = accuracy_score(y_val, rf_pred_val)

print(f"✅ Random Forest Trained!")
print(f"   Training Accuracy: {rf_train_acc:.4f}")
print(f"   Validation Accuracy: {rf_val_acc:.4f}")

In [ ]:
# 6.2 Train Gradient Boosting Classifier (Best Model)
print("Training Gradient Boosting Classifier...")
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
gb_model.fit(X_train_scaled, y_train)

# Predictions
gb_pred_train = gb_model.predict(X_train_scaled)
gb_pred_val = gb_model.predict(X_val_scaled)

# Evaluation
gb_train_acc = accuracy_score(y_train, gb_pred_train)
gb_val_acc = accuracy_score(y_val, gb_pred_val)

print(f"✅ Gradient Boosting Trained!")
print(f"   Training Accuracy: {gb_train_acc:.4f}")
print(f"   Validation Accuracy: {gb_val_acc:.4f}")
print(f"\n🏆 Gradient Boosting is our best model!")

In [ ]:
# 6.3 Detailed Evaluation Metrics
print("="*60)
print("GRADIENT BOOSTING - VALIDATION SET PERFORMANCE")
print("="*60)

print(f"\nAccuracy: {gb_val_acc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_val, gb_pred_val, target_names=['Did Not Survive', 'Survived']))

# Confusion Matrix
cm = confusion_matrix(y_val, gb_pred_val)
print(f"\nConfusion Matrix:")
print(cm)

In [ ]:
# 6.4 Visualize Model Comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy comparison
models = ['Random Forest', 'Gradient Boosting']
train_acc = [rf_train_acc, gb_train_acc]
val_acc = [rf_val_acc, gb_val_acc]

x = np.arange(len(models))
width = 0.35

axes[0].bar(x - width/2, train_acc, width, label='Training', color='skyblue')
axes[0].bar(x + width/2, val_acc, width, label='Validation', color='coral')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Comparison - Accuracy', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models)
axes[0].legend()
axes[0].set_ylim([0.7, 1.0])

# Confusion Matrix for best model
cm_gb = confusion_matrix(y_val, gb_pred_val)
sns.heatmap(cm_gb, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Did Not Survive', 'Survived'],
            yticklabels=['Did Not Survive', 'Survived'])
axes[1].set_title('Gradient Boosting - Confusion Matrix', fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

## Section 7: Feature Importance Analysis

In [ ]:
# 7.1 Get Feature Importance from Gradient Boosting
feature_importance = pd.DataFrame({
    'Feature': X_train_full.columns,
    'Importance': gb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n" + "="*50)
print("FEATURE IMPORTANCE (Gradient Boosting)")
print("="*50)
print(feature_importance.to_string(index=False))

# 7.2 Visualize Feature Importance
fig, ax = plt.subplots(figsize=(10, 6))

ax.barh(feature_importance['Feature'], feature_importance['Importance'], color='steelblue')
ax.set_xlabel('Importance')
ax.set_title('Feature Importance - Gradient Boosting Model', fontweight='bold', fontsize=12)
ax.invert_yaxis()

# Add value labels
for i, v in enumerate(feature_importance['Importance']):
    ax.text(v + 0.01, i, f'{v:.3f}', va='center')

plt.tight_layout()
plt.show()

## Section 8: Generate Predictions for Kaggle Submission

In [ ]:
# 8.1 Train on full training data for final submission
print("Training final model on full training data...")
X_train_full_scaled = scaler.fit_transform(X_train_full)
gb_final = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
gb_final.fit(X_train_full_scaled, y_train_full)

print("✅ Final model trained on full training set!")

In [ ]:
# 8.2 Generate predictions on test set
X_test_full_scaled = scaler.transform(X_test)
test_predictions = gb_final.predict(X_test_full_scaled)

print(f"✅ Test Predictions Generated!")
print(f"   Total Predictions: {len(test_predictions)}")
print(f"   Predicted Survived: {(test_predictions == 1).sum()} ({(test_predictions == 1).sum() / len(test_predictions) * 100:.1f}%)")
print(f"   Predicted Did Not Survive: {(test_predictions == 0).sum()} ({(test_predictions == 0).sum() / len(test_predictions) * 100:.1f}%)")

In [ ]:
# 8.3 Create Kaggle Submission File
submission_df = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': test_predictions
})

# Save submission
submission_df.to_csv('submission.csv', index=False)

print("✅ Submission File Created: submission.csv")
print(f"\nFirst 10 rows of submission:")
print(submission_df.head(10))
print(f"\nLast 10 rows of submission:")
print(submission_df.tail(10))

## Section 9: Model Performance Summary

In [ ]:
print("\n" + "="*60)
print("FINAL MODEL PERFORMANCE SUMMARY")
print("="*60)

print(f"\n🤖 MODEL: Gradient Boosting Classifier")
print(f"   - n_estimators: 100")
print(f"   - learning_rate: 0.1")
print(f"   - max_depth: 5")

print(f"\n📊 VALIDATION METRICS:")
print(f"   - Accuracy: {gb_val_acc:.4f}")
print(f"   - Training Accuracy: {gb_train_acc:.4f}")

print(f"\n🏆 TOP 5 FEATURES:")
for idx, row in feature_importance.head(5).iterrows():
    print(f"   {idx+1}. {row['Feature']}: {row['Importance']:.3f}")

print(f"\n📁 TEST PREDICTIONS:")
print(f"   - Total Test Passengers: {len(test_predictions)}")
print(f"   - Predicted Survived: {(test_predictions == 1).sum()}")
print(f"   - Predicted Did Not Survive: {(test_predictions == 0).sum()}")

print(f"\n📤 SUBMISSION FILE: submission.csv")
print(f"   - Format: CSV with PassengerId and Survived columns")
print(f"   - Ready for Kaggle submission!")

print(f"\n🎯 KAGGLE SCORE: 0.80143 (Ranked 812)")
print("\n" + "="*60)

## Next Steps to Improve Score

1. **Hyperparameter Tuning**: Use GridSearchCV to find optimal parameters
2. **Feature Engineering**: Create more meaningful features from existing data
3. **Ensemble Methods**: Try voting or stacking with multiple models
4. **Cross-Validation**: Implement k-fold CV for better generalization
5. **Class Weighting**: Handle class imbalance with weighted models
6. **Feature Selection**: Use RFE or mutual information to select best features

---

**Created**: 2026-05-10  
**Author**: Steve Coleman (@richsteve17)  
**Repository**: [titanic-competition-](https://github.com/richsteve17/titanic-competition-)